# SentimentForge – Transformer Fine-Tuning

## Objective
Fine-tune a pretrained DistilBERT model for binary sentiment
classification using the IMDb movie review dataset.

## Model
DistilBERT (Hugging Face Transformers)

## Framework
PyTorch

## Dataset
IMDb 50K Movie Reviews

## Tasks
1. Understand Transformer architecture and attention.
2. Load pretrained DistilBERT tokenizer and model.
3. Tokenize movie reviews.
4. Prepare PyTorch datasets and DataLoaders.
5. Fine-tune DistilBERT.
6. Evaluate model performance.
7. Compare with RNN, LSTM, BiLSTM and GRU.
8. Save the trained model and experiment results.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

print("PyTorch:", torch.__version__)
print("Libraries imported successfully!")

PyTorch: 2.11.0+cu130
Libraries imported successfully!


In [ ]:
from google.colab import drive
import os
import pandas as pd
import json

# Mount Google Drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/SentimentForge"

DATASET_PATH = os.path.join(
    PROJECT_DIR, "data", "raw", "IMDB Dataset.csv"
)

MODEL_DIR = os.path.join(PROJECT_DIR, "models")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")

CONFIG_PATH = os.path.join(
    MODEL_DIR, "preprocessing_config.json"
)

# Ensure the directories exist or read directly if already present
df = pd.read_csv(DATASET_PATH)

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

print("Dataset shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nPreprocessing configuration:")
print(config)

Mounted at /content/drive
Dataset shape: (50000, 2)

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Preprocessing configuration:
{'max_words': 30000, 'max_length': 300, 'padding': 'post', 'truncating': 'post', 'oov_token': '<OOV>', 'negative_label': 0, 'positive_label': 1}


In [ ]:
# Convert sentiment labels into numerical values

label_map = {
    "negative": 0,
    "positive": 1
}

df["label"] = df["sentiment"].map(label_map)

# Separate reviews and labels

X = df["review"].values
y = df["label"].values

# First split: 80% training, 20% temporary

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining labels:")
print(pd.Series(y_train).value_counts())

print("\nValidation labels:")
print(pd.Series(y_val).value_counts())

print("\nTesting labels:")
print(pd.Series(y_test).value_counts())

Training samples: 40000
Validation samples: 5000
Testing samples: 5000

Training labels:
1    20000
0    20000
Name: count, dtype: int64

Validation labels:
0    2500
1    2500
Name: count, dtype: int64

Testing labels:
0    2500
1    2500
Name: count, dtype: int64


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)
print("Unknown token:", tokenizer.unk_token)
print("Special tokens:", tokenizer.special_tokens_map)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522
Padding token: [PAD]
Padding token ID: 0
Unknown token: [UNK]
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [ ]:
sample_review = "The movie was not good, but the acting was fantastic!"

tokens = tokenizer.tokenize(sample_review)

token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original review:")
print(sample_review)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Original review:
The movie was not good, but the acting was fantastic!

Tokens:
['the', 'movie', 'was', 'not', 'good', ',', 'but', 'the', 'acting', 'was', 'fantastic', '!']

Token IDs:
[1996, 3185, 2001, 2025, 2204, 1010, 2021, 1996, 3772, 2001, 10392, 999]


In [ ]:
encoded_sample = tokenizer(
    sample_review,
    add_special_tokens=True
)

print("Input IDs:")
print(encoded_sample["input_ids"])

print("\nAttention Mask:")
print(encoded_sample["attention_mask"])

print("\nDecoded tokens:")
print(
    tokenizer.convert_ids_to_tokens(
        encoded_sample["input_ids"]
    )
)

Input IDs:
[101, 1996, 3185, 2001, 2025, 2204, 1010, 2021, 1996, 3772, 2001, 10392, 999, 102]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded tokens:
['[CLS]', 'the', 'movie', 'was', 'not', 'good', ',', 'but', 'the', 'acting', 'was', 'fantastic', '!', '[SEP]']


In [ ]:
from tqdm.auto import tqdm

token_lengths = []

for review in tqdm(X_train, desc="Calculating token lengths"):

    encoded = tokenizer(
        review,
        add_special_tokens=True,
        truncation=False
    )

    token_lengths.append(len(encoded["input_ids"]))

token_lengths = np.array(token_lengths)

print("Token length statistics:")
print(pd.Series(token_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

Calculating token lengths:   0%|          | 0/40000 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (727 > 512). Running this sequence through the model will result in indexing errors


Token length statistics:
count    40000.000000
mean       310.527450
std        230.589634
min         11.000000
50%        231.500000
75%        379.000000
90%        611.000000
95%        796.000000
99%       1198.000000
max       3157.000000
dtype: float64


In [ ]:
candidate_lengths = [256, 384, 512]

print("Sequence Length Coverage")
print("-" * 45)

for length in candidate_lengths:

    retained = np.sum(token_lengths <= length)
    truncated = np.sum(token_lengths > length)

    retained_pct = (retained / len(token_lengths)) * 100
    truncated_pct = (truncated / len(token_lengths)) * 100

    print(f"\nMAX_LEN: {length}")
    print(f"Reviews fully retained: {retained_pct:.2f}%")
    print(f"Reviews truncated:      {truncated_pct:.2f}%")

Sequence Length Coverage
---------------------------------------------

MAX_LEN: 256
Reviews fully retained: 56.15%
Reviews truncated:      43.85%

MAX_LEN: 384
Reviews fully retained: 75.62%
Reviews truncated:      24.38%

MAX_LEN: 512
Reviews fully retained: 85.55%
Reviews truncated:      14.46%


In [ ]:
MAX_LEN = 512
BATCH_SIZE = 8

print("Maximum sequence length:", MAX_LEN)
print("Batch size:", BATCH_SIZE)
print("Model maximum length:", tokenizer.model_max_length)

Maximum sequence length: 512
Batch size: 8
Model maximum length: 512


In [ ]:
sample_encoding = tokenizer(
    sample_review,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("Input IDs shape:", sample_encoding["input_ids"].shape)
print("Attention mask shape:", sample_encoding["attention_mask"].shape)

print("\nActual tokens:", sample_encoding["attention_mask"].sum().item())
print("Padding tokens:", (sample_encoding["attention_mask"] == 0).sum().item())

Input IDs shape: torch.Size([1, 512])
Attention mask shape: torch.Size([1, 512])

Actual tokens: 14
Padding tokens: 498


In [ ]:
def tokenize_reviews(reviews, tokenizer, max_length):

    encodings = tokenizer(
        list(reviews),
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return encodings


train_encodings = tokenize_reviews(X_train, tokenizer, MAX_LEN)
val_encodings = tokenize_reviews(X_val, tokenizer, MAX_LEN)
test_encodings = tokenize_reviews(X_test, tokenizer, MAX_LEN)

print("Tokenization completed!")

print("\nTraining input shape:", train_encodings["input_ids"].shape)
print("Validation input shape:", val_encodings["input_ids"].shape)
print("Testing input shape:", test_encodings["input_ids"].shape)

Tokenization completed!

Training input shape: torch.Size([40000, 512])
Validation input shape: torch.Size([5000, 512])
Testing input shape: torch.Size([5000, 512])


In [ ]:
class IMDbDataset(Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

        return item

In [ ]:
train_dataset = IMDbDataset(train_encodings, y_train)
val_dataset = IMDbDataset(val_encodings, y_val)
test_dataset = IMDbDataset(test_encodings, y_test)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 40000
Validation dataset: 5000
Testing dataset: 5000


In [ ]:
sample = train_dataset[0]

print("Available keys:", sample.keys())
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["labels"])

Available keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([512])
Attention mask shape: torch.Size([512])
Label: tensor(1)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 5000
Validation batches: 625
Testing batches: 625


In [ ]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape, value.dtype)

input_ids torch.Size([8, 512]) torch.int64
attention_mask torch.Size([8, 512]) torch.int64
labels torch.Size([8]) torch.int64


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

device = torch.device("cuda")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = model.to(device)

print("Model loaded successfully!")
print("Device:", device)
print("Number of labels:", model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Device: cuda
Number of labels: 2


In [ ]:
batch = next(iter(train_loader))

# Move batch tensors to GPU
batch = {
    key: value.to(device)
    for key, value in batch.items()
}


In [ ]:
outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"]
)

loss = outputs.loss
logits = outputs.logits

from torch.optim import AdamW

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

In [ ]:
total_training_steps = len(train_loader) * EPOCHS

warmup_steps = int(0.1 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Total training steps: 15000
Warmup steps: 1500


In [ ]:
batch = next(iter(train_loader))

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 0.6818156242370605
Logits shape: torch.Size([8, 2])


In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, scheduler, device):

    model.train()

    total_loss = 0
    correct_predictions = 0
    total_predictions = 0

    progress_bar = tqdm(loader, desc="Training")

    for batch in progress_bar:

        # Move batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(**batch)

        loss = outputs.loss
        logits = outputs.logits

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Update learning rate
        scheduler.step()

        # Track loss
        total_loss += loss.item()

        # Calculate predictions
        predictions = torch.argmax(logits, dim=1)

        correct_predictions += (
            (predictions == batch["labels"]).sum().item()
        )

        total_predictions += batch["labels"].size(0)

        progress_bar.set_postfix(
            loss=loss.item()
        )

    average_loss = total_loss / len(loader)

    accuracy = correct_predictions / total_predictions

    return average_loss, accuracy

In [ ]:
def evaluate_model(model, loader, device):

    model.eval()

    total_loss = 0

    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for batch in tqdm(loader, desc="Evaluating"):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            # Convert logits into probabilities
            probabilities = torch.softmax(logits, dim=1)

            # Select the predicted class
            predictions = torch.argmax(logits, dim=1)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                batch["labels"].cpu().numpy()
            )

            # Probability of positive sentiment (class 1)
            all_probabilities.extend(
                probabilities[:, 1].cpu().numpy()
            )

    average_loss = total_loss / len(loader)

    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)

    roc_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(**batch)
    loss = outputs.loss

scaler.scale(loss).backward()

scaler.step(optimizer)
scaler.update()

scheduler.step()

In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_roc_auc": []
}

best_val_loss = float("inf")

In [ ]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, device
    )

    val_metrics = evaluate_model(
        model, val_loader, device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_roc_auc"].append(val_metrics["roc_auc"])

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Loss: {val_metrics['loss']:.4f}")
    print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Validation F1: {val_metrics['f1']:.4f}")
    print(f"Validation ROC-AUC: {val_metrics['roc_auc']:.4f}")

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        torch.save(
            model.state_dict(),
            os.path.join(MODEL_DIR, "distilbert_best.pth")
        )

        print("Best model checkpoint saved!")


Epoch 1/3


Training:   0%|          | 0/5000 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]

Train Loss: 0.2555
Train Accuracy: 0.8879
Validation Loss: 0.1726
Validation Accuracy: 0.9334
Validation F1: 0.9346
Validation ROC-AUC: 0.9826
Best model checkpoint saved!

Epoch 2/3


Training:   0%|          | 0/5000 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]

Train Loss: 0.1203
Train Accuracy: 0.9569
Validation Loss: 0.1565
Validation Accuracy: 0.9398
Validation F1: 0.9405
Validation ROC-AUC: 0.9856
Best model checkpoint saved!

Epoch 3/3


Training:   0%|          | 0/5000 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]

Train Loss: 0.0474
Train Accuracy: 0.9858
Validation Loss: 0.1901
Validation Accuracy: 0.9428
Validation F1: 0.9433
Validation ROC-AUC: 0.9857


In [ ]:
best_model_path = os.path.join(
    MODEL_DIR, "distilbert_best.pth"
)

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model.eval()

test_metrics = evaluate_model(
    model,
    test_loader,
    device
)

print("\nDistilBERT Test Results")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]


DistilBERT Test Results
loss: 0.1583
accuracy: 0.9412
precision: 0.9380
recall: 0.9448
f1: 0.9414
roc_auc: 0.9851


In [ ]:
DISTILBERT_DIR = os.path.join(
    MODEL_DIR, "distilbert_finetuned"
)

model.save_pretrained(DISTILBERT_DIR)
tokenizer.save_pretrained(DISTILBERT_DIR)

print("DistilBERT model and tokenizer saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT model and tokenizer saved!


In [ ]:
experiment_results = {
    "model": "DistilBERT",
    "pretrained_model": MODEL_NAME,
    "best_epoch": 2,
    "epochs_trained": EPOCHS,
    "max_length": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "test_metrics": test_metrics,
    "training_history": history
}

results_path = os.path.join(
    REPORT_DIR, "distilbert_experiment.json"
)

with open(results_path, "w") as f:
    json.dump(experiment_results, f, indent=4)

print("Experiment results saved!")

Experiment results saved!
